# Assignment 4 — measured experiment: Seq-LoRA vs Experience Replay vs Online EWC (+ Joint)
**Research Methods and Tools · SE-2418** · Mirzanov Temirlan · Turemuratov Galymzhan · Abdikul Bakdaulet

Real data (AG News → SST-2 → Emotion, 1,000 / 200 / 800 per task, seed 1234) and the real model
(`Qwen/Qwen2.5-0.5B` + LoRA r=8 on q/k/v/o). 19 runs exactly as in the frozen protocol:
5 EWC λ-calibration runs (seed 1000, validation data) · 4 conditions × seeds 0/1/2 · ER 5% and 20% sensitivity (seed 0).

**How to run:** `Runtime → Change runtime type → T4 GPU` → `Runtime → Run all` → allow Google Drive.
Every finished run is saved to `MyDrive/rmt_a4_real/`; if Colab disconnects, press *Run all* again — finished
runs are skipped. Expected time on a free T4: about 1.5–2 hours. At the end `rmt_a4_real_results.zip` downloads
automatically — send that file back.

In [ ]:
!pip -q install -U peft datasets
!pip -q uninstall -y torchao   # Colab ships torchao 0.10, which new peft rejects; it is not needed here

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re, json, time, math, random, hashlib, platform, sys, zipfile, csv, io
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn.functional as Fn

SMOKE_TEST = bool(int(os.environ.get("SMOKE_TEST", "0")))   # local code check only (tiny random model, synthetic data)

CFG = dict(
    model_name="Qwen/Qwen2.5-0.5B",
    tasks=["ag_news", "sst2", "emotion"],
    n_train=1000, n_val=200, n_test=800, data_seed=1234,
    max_text_tokens=96, max_len=160,
    lora_r=8, lora_alpha=16, lora_dropout=0.05, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lr=2e-4, epochs=2, batch_size=16, warmup_ratio=0.1, weight_decay=0.0, grad_clip=1.0,
    eval_batch_size=64,
    replay_fraction=0.10, sensitivity_fractions=[0.05, 0.20], sensitivity_seed=0,
    ewc_fisher_samples=200, ewc_lambda_grid=[0.01, 0.1, 1.0, 10.0, 100.0], calibration_seed=1000,
    seeds=[0, 1, 2],
    bootstrap_resamples=10000, stats_seed=2026,
    results_dir="/content/drive/MyDrive/rmt_a4_real",
)
if SMOKE_TEST:
    CFG.update(n_train=40, n_val=12, n_test=20, epochs=1, batch_size=8, ewc_fisher_samples=6,
               ewc_lambda_grid=[0.1, 10.0], seeds=[0, 1], eval_batch_size=16, bootstrap_resamples=500,
               sensitivity_fractions=[0.05, 0.20], results_dir=os.environ.get("RESULTS_DIR", "/tmp/a4_smoke"))
if not os.path.isdir("/content/drive") and not SMOKE_TEST:
    CFG["results_dir"] = "/content/rmt_a4_real"
RES = CFG["results_dir"]
for d in ("runs", "figures", "tables", "data"): os.makedirs(f"{RES}/{d}", exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
print("device:", DEVICE, "| results ->", RES)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

## 1. Environment record

In [ ]:
import transformers, peft, importlib
for _k in [k for k in list(sys.modules) if k.startswith("torchao")]: del sys.modules[_k]
importlib.invalidate_caches()
try:
    import peft.import_utils as _iu; _iu.is_torchao_available.cache_clear()
except Exception: pass
def env_info():
    info = dict(python=sys.version.split()[0], platform=platform.platform(), torch=torch.__version__,
                transformers=transformers.__version__, peft=peft.__version__, numpy=np.__version__,
                device=DEVICE, cuda=torch.version.cuda,
                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                gpu_mem_GB=round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if torch.cuda.is_available() else None,
                smoke_test=SMOKE_TEST, started_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
    try:
        import datasets; info["datasets"] = datasets.__version__
    except Exception: pass
    return info
ENV = env_info()
json.dump(ENV, open(f"{RES}/env.json", "w"), indent=2)
print(json.dumps(ENV, indent=2))

## 2. Frozen dataset (6,000 records)
Sources (pinned revisions from the repository manifest): AG News (`fancyzhx/ag_news`), SST-2 (`nyu-mll/glue`, `sst2`),
Emotion (`dair-ai/emotion`, `split`). AG News / SST-2: train + validation from the official train split; test from the
official test split (AG News) or the official validation split (SST-2). Emotion: official train / validation / test.
Fixed-seed random sampling without replacement (seed 1234), whitespace normalisation, global de-duplication after
case-folding. Several standard implementations of this sampling rule are tried and compared with the class counts and
split hashes recorded in `data/manifests` of the repository; the first exact match is used, otherwise the canonical
rule is used and the new hashes are reported.

In [ ]:
LABELS = {1: ("World", "Sports", "Business", "Sci/Tech"), 2: ("Negative", "Positive"),
          3: ("Sadness", "Joy", "Love", "Anger", "Fear", "Surprise")}
TASK_NAMES = {1: "AG News", 2: "SST-2", 3: "Emotion"}
TASK_KEY = {1: "ag_news", 2: "sst2", 3: "emotion"}; TASK_ID = {v: k for k, v in TASK_KEY.items()}
SOURCES = {1: dict(hf="fancyzhx/ag_news", conf=None, rev="eb185aade064a813bc0b7f42de02595523103ca4", text="text",
                   plan={"train": "train", "validation": "train", "test": "test"}),
           2: dict(hf="nyu-mll/glue", conf="sst2", rev="bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c", text="sentence",
                   plan={"train": "train", "validation": "train", "test": "validation"}),
           3: dict(hf="dair-ai/emotion", conf="split", rev="cab853a1dbdf4c42c2b3ef2173804746df8825fe", text="text",
                   plan={"train": "train", "validation": "validation", "test": "test"})}
EXPECTED_SPLIT_HASH = {(1, "train"): "fa2736a88fa1bb395e97049ef44194b666a20f647d7b7358a2831d6cd1c54ae6",
    (1, "validation"): "2e475029464b6d9cc22842810c9d10fab7922bb3e8e857ae850dac1d03a401cc",
    (1, "test"): "e9ff0317dabbdc964c20c0524952eeeec21a17d9d5d8c895bbfd9d5e66f70b01",
    (2, "train"): "47c8c7f6f3cd7c6770567e871224d7c4720259b584cb23c09f669ee02d76fd5e",
    (2, "validation"): "569263a790c6a022a0fc2da7fdaa92404639947d6ee5faf30f657222d6fbae35",
    (2, "test"): "d62a60b5acaf6e46e2ffdb8ab54610031b05d26fa223177f0eb83fdda58f2fb4",
    (3, "train"): "dd70e3240db4021fa7f38de6383d4faa8ec5a54c236ff2583f85668da8651835",
    (3, "validation"): "f0e77d8f766abe75a2b50ded95631b6a22361a9703efd22d4f119fcbea7fefeb",
    (3, "test"): "2c8ffcab665b767d8371c852b7df10e1033904500eac73251c1e88ca9aa526d7"}
EXPECTED_CLASS_COUNTS = {
    (1, "test"): [218, 167, 204, 211], (1, "train"): [230, 262, 254, 254], (1, "validation"): [39, 47, 59, 55],
    (2, "test"): [396, 404], (2, "train"): [442, 558], (2, "validation"): [82, 118],
    (3, "test"): [243, 277, 65, 96, 88, 31], (3, "train"): [309, 342, 78, 131, 119, 21],
    (3, "validation"): [57, 70, 12, 33, 20, 8]}
SPLITS = ("train", "validation", "test")
N_SPLIT = {"train": CFG["n_train"], "validation": CFG["n_val"], "test": CFG["n_test"]}

def normalize_text(t): return re.sub(r"\s+", " ", str(t)).strip()
def text_hash(t): return hashlib.sha256(normalize_text(t).casefold().encode("utf-8")).hexdigest()

def load_sources():
    from datasets import load_dataset
    out = {}
    for task, s in SOURCES.items():
        args = (s["hf"], s["conf"]) if s["conf"] else (s["hf"],)
        try:
            ds = load_dataset(*args, revision=s["rev"])
        except Exception as e:
            print(f"{s['hf']}: pinned revision unavailable ({type(e).__name__}); loading default revision")
            ds = load_dataset(*args)
        out[task] = {sp: [(normalize_text(r[s["text"]]), int(r["label"])) for r in ds[sp]] for sp in set(s["plan"].values())}
        print(TASK_NAMES[task], {sp: len(v) for sp, v in out[task].items()})
    return out

def smoke_sources():
    rng = random.Random(0); words = "the a of and to in is it that was for on with as by this not very good bad".split()
    def rows(task, n): return [(" ".join(rng.choice(words) for _ in range(rng.randint(4, 14))) + f" {LABELS[task][y].lower()} {i}", y)
                                for i in range(n) for y in [rng.randrange(len(LABELS[task]))]]
    return {1: {"train": rows(1, 300), "test": rows(1, 100)}, 2: {"train": rows(2, 300), "validation": rows(2, 100)},
            3: {"train": rows(3, 300), "validation": rows(3, 100), "test": rows(3, 100)}}

def _perm(gen, n):
    kind, state = gen
    if kind == "RandomState": return list(state.permutation(n))
    if kind == "default_rng": return list(state.permutation(n))
    if kind == "random": idx = list(range(n)); state.shuffle(idx); return idx
    raise ValueError(kind)

def _new_gen(kind, seed):
    return (kind, np.random.RandomState(seed) if kind == "RandomState" else
                  np.random.default_rng(seed) if kind == "default_rng" else random.Random(seed))

def sample_dataset(src, kind="RandomState", scope="global", order="test_first"):
    """Fixed-seed sampling without replacement + global de-duplication (normalised, case-folded text)."""
    seen, out = set(), {}
    gen = _new_gen(kind, CFG["data_seed"])
    for task in (1, 2, 3):
        if scope == "per_task": gen = _new_gen(kind, CFG["data_seed"])
        plan = SOURCES[task]["plan"]
        # group target splits by source split; order of groups
        groups = defaultdict(list)
        for sp in SPLITS: groups[plan[sp]].append(sp)
        src_order = sorted(groups, key=lambda s: (0 if ("test" in groups[s]) == (order == "test_first") else 1, s))
        for s in src_order:
            rows = src[task][s]; take = []
            for i in _perm(gen, len(rows)):
                t, y = rows[i]
                if not t or not (0 <= y < len(LABELS[task])): continue
                h = text_hash(t)
                if h in seen: continue
                seen.add(h); take.append((int(i), t, y, h, s))
                if len(take) == sum(N_SPLIT[sp] for sp in groups[s]): break
            pos = 0
            for sp in [x for x in SPLITS if x in groups[s]]:
                out[(task, sp)] = take[pos:pos + N_SPLIT[sp]]; pos += N_SPLIT[sp]
    return out

def class_counts(sel):
    return {k: [sum(1 for r in v if r[2] == c) for c in range(len(LABELS[k[0]]))] for k, v in sel.items()}

def split_hash(rows, id_style, sort):
    rs = sorted(rows, key=lambda r: r[0]) if sort else rows
    content = "\n".join(f"{rid}\t{r[3]}" for r in rs for rid in [id_style(r)])
    return hashlib.sha256(content.encode("utf-8")).hexdigest()

SRC = smoke_sources() if SMOKE_TEST else load_sources()
DATA_MATCH = dict(tried=[], selected=None, class_counts_match=False, split_hash_match=False)
SEL = None
for kind in ("RandomState", "default_rng", "random"):
    for scope in ("global", "per_task"):
        for order in ("test_first", "train_first"):
            sel = sample_dataset(SRC, kind, scope, order)
            ok = all(len(v) == N_SPLIT[k[1]] for k, v in sel.items())
            cc_match = ok and class_counts(sel) == EXPECTED_CLASS_COUNTS
            DATA_MATCH["tried"].append(dict(kind=kind, scope=scope, order=order, complete=ok, class_counts_match=cc_match))
            if cc_match and SEL is None:
                SEL, DATA_MATCH["selected"], DATA_MATCH["class_counts_match"] = sel, (kind, scope, order), True
if SEL is None:   # canonical rule (Assignment 3 notebook): one RandomState(1234), test sampled first
    SEL = sample_dataset(SRC, "RandomState", "global", "test_first"); DATA_MATCH["selected"] = ("RandomState", "global", "test_first")
def make_ids(task, sp):
    return {"frozen_split": lambda r: f"T{task}_{sp}_{r[0]:07d}", "source_split": lambda r: f"T{task}_{r[4]}_{r[0]:07d}"}
if DATA_MATCH["class_counts_match"]:
    for idn in ("frozen_split", "source_split"):
        for sort in (False, True):
            if all(split_hash(SEL[(t, sp)], make_ids(t, sp)[idn], sort) == EXPECTED_SPLIT_HASH[(t, sp)] for t in (1, 2, 3) for sp in SPLITS):
                DATA_MATCH.update(split_hash_match=True, id_style=idn, sorted=sort); break
        if DATA_MATCH["split_hash_match"]: break
ID_STYLE = DATA_MATCH.get("id_style", "frozen_split"); SORT = DATA_MATCH.get("sorted", False)

RECORDS = []   # rows of the frozen CSV, repository schema
for t in (1, 2, 3):
    for sp in SPLITS:
        rows = sorted(SEL[(t, sp)], key=lambda r: r[0]) if SORT else SEL[(t, sp)]
        for r in rows:
            RECORDS.append(dict(split=sp, task=t, source_index=r[0], label=r[2], text=r[1], task_name=TASK_NAMES[t],
                                dataset=SOURCES[t]["hf"], source_split=r[4], label_name=LABELS[t][r[2]],
                                record_id=make_ids(t, sp)[ID_STYLE](r), text_sha256=r[3]))
cols = ["split", "task", "source_index", "label", "text", "task_name", "dataset", "source_split", "label_name", "record_id", "text_sha256"]
buf = io.StringIO(); w = csv.DictWriter(buf, fieldnames=cols, lineterminator="\n"); w.writeheader(); w.writerows(RECORDS)
csv_bytes = buf.getvalue().encode("utf-8")
open(f"{RES}/data/assignment4_real_6000.csv", "wb").write(csv_bytes)
SPLIT_HASHES = {f"{t}/{sp}": hashlib.sha256("\n".join(f"{r['record_id']}\t{r['text_sha256']}" for r in RECORDS
                                            if r["task"] == t and r["split"] == sp).encode()).hexdigest() for t in (1, 2, 3) for sp in SPLITS}
DATA_MANIFEST = dict(csv_sha256=hashlib.sha256(csv_bytes).hexdigest(), total=len(RECORDS),
                     unique_texts=len({r["text_sha256"] for r in RECORDS}), split_hashes=SPLIT_HASHES,
                     class_counts={f"{t}/{sp}": c for (t, sp), c in class_counts(SEL).items()},
                     matches_repository_manifest=DATA_MATCH)
json.dump(DATA_MANIFEST, open(f"{RES}/data/dataset_manifest_measured.json", "w"), indent=2)
print("CSV SHA-256:", DATA_MANIFEST["csv_sha256"], "| unique texts:", DATA_MANIFEST["unique_texts"])
print("matches repository class counts:", DATA_MATCH["class_counts_match"], "| split hashes:", DATA_MATCH["split_hash_match"],
      "| rule:", DATA_MATCH["selected"])
assert DATA_MANIFEST["unique_texts"] == len(RECORDS) == sum(N_SPLIT.values()) * 3

DATA = {TASK_KEY[t]: {sp: [r for r in RECORDS if r["task"] == t and r["split"] == sp] for sp in SPLITS} for t in (1, 2, 3)}

## 3. Tokenizer, prompts (repository templates) and encoding

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
PROMPTS = {1: ("Classify the news topic.", ("World", "Sports", "Business", "Technology")),
           2: ("Classify the sentiment.", ("Negative", "Positive")),
           3: ("Classify the expressed emotion.", ("Sadness", "Joy", "Love", "Anger", "Fear", "Surprise"))}

def build_smoke_tokenizer():
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
    from transformers import PreTrainedTokenizerFast
    corpus = [r["text"] for r in RECORDS] + [f"Task: {i}\nOptions: {', '.join(l)}\nText: Answer:" for i, l in PROMPTS.values()]
    tk = Tokenizer(models.BPE(unk_token="<unk>"))
    tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False); tk.decoder = decoders.ByteLevel()
    tk.train_from_iterator(corpus * 3, trainers.BpeTrainer(vocab_size=500, special_tokens=["<unk>", "<|endoftext|>"],
                                                           initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))
    return PreTrainedTokenizerFast(tokenizer_object=tk, eos_token="<|endoftext|>", pad_token="<|endoftext|>", unk_token="<unk>")

tok = build_smoke_tokenizer() if SMOKE_TEST else AutoTokenizer.from_pretrained(CFG["model_name"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
EOS = tok.eos_token_id

def prompt_ids(task_id, text):
    t_ids = tok(text, add_special_tokens=False).input_ids[:CFG["max_text_tokens"]]
    instr, labs = PROMPTS[task_id]
    p = f"Task: {instr}\nOptions: {', '.join(labs)}\nText: {tok.decode(t_ids).strip()}\nAnswer:"
    return tok(p, add_special_tokens=True).input_ids

TARGETS = {t: [tok(" " + l, add_special_tokens=False).input_ids + [EOS] for l in PROMPTS[t][1]] for t in (1, 2, 3)}
PROMPT_CACHE = {}
def cached_prompt(r):
    k = r["record_id"] + r["split"]
    if k not in PROMPT_CACHE: PROMPT_CACHE[k] = prompt_ids(r["task"], r["text"])
    return PROMPT_CACHE[k]

def encode(r):
    p, a = cached_prompt(r), TARGETS[r["task"]][r["label"]]
    p = p[: CFG["max_len"] - len(a)]
    return dict(ids=p + a, tgt=[0] * len(p) + [1] * len(a))

card = []
for tk_, d in DATA.items():
    for sp in SPLITS:
        L = [len(encode(r)["ids"]) for r in d[sp]]
        card.append(dict(task=tk_, split=sp, n=len(d[sp]), class_counts=dict(Counter(r["label_name"] for r in d[sp])),
                         mean_tokens=round(float(np.mean(L)), 1), max_tokens=int(np.max(L))))
json.dump(card, open(f"{RES}/data/data_card.json", "w"), indent=2)
for c in card: print(c)
print("\nExample prompt:\n" + tok.decode(encode(DATA["ag_news"]["train"][0])["ids"]))

## 4. Model, loss on answer tokens, constrained label-likelihood evaluation

In [ ]:
from peft import LoraConfig, get_peft_model

def load_model(seed):
    set_seed(seed)
    if SMOKE_TEST:
        from transformers import Qwen2Config, Qwen2ForCausalLM
        torch.manual_seed(123)
        base = Qwen2ForCausalLM(Qwen2Config(vocab_size=len(tok), hidden_size=64, intermediate_size=128, num_hidden_layers=2,
                                            num_attention_heads=4, num_key_value_heads=2, max_position_embeddings=256,
                                            tie_word_embeddings=True))
        set_seed(seed)
    else:
        try: base = AutoModelForCausalLM.from_pretrained(CFG["model_name"], dtype=torch.float32)
        except TypeError: base = AutoModelForCausalLM.from_pretrained(CFG["model_name"], torch_dtype=torch.float32)
    lcfg = LoraConfig(r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"], lora_dropout=CFG["lora_dropout"],
                      target_modules=CFG["target_modules"], bias="none", task_type="CAUSAL_LM")
    return get_peft_model(base, lcfg).to(DEVICE)

def trainable(model): return {n: p for n, p in model.named_parameters() if p.requires_grad}

def collate(batch):
    T = max(len(e["ids"]) for e in batch)
    ids = torch.full((len(batch), T), tok.pad_token_id, dtype=torch.long)
    att = torch.zeros((len(batch), T), dtype=torch.long); tgt = torch.zeros((len(batch), T), dtype=torch.bool)
    for i, e in enumerate(batch):
        n = len(e["ids"]); ids[i, :n] = torch.tensor(e["ids"]); att[i, :n] = 1; tgt[i, :n] = torch.tensor(e["tgt"], dtype=torch.bool)
    return ids.to(DEVICE), att.to(DEVICE), tgt.to(DEVICE)

def target_logprobs(model, ids, att, tgt):
    base = model.get_base_model()
    with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
        h = base.model(input_ids=ids, attention_mask=att).last_hidden_state
    m = tgt[:, 1:]; hs = h[:, :-1][m]
    with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
        logits = base.lm_head(hs)
    logp = Fn.log_softmax(logits.float(), dim=-1)
    rows = torch.arange(ids.size(0), device=ids.device).unsqueeze(1).expand_as(m)[m]
    return logp.gather(1, ids[:, 1:][m].unsqueeze(1)).squeeze(1), rows

def lm_loss(model, records):
    ids, att, tgt = collate([encode(r) for r in records])
    lp, _ = target_logprobs(model, ids, att, tgt)
    return -lp.mean()

def macro_f1(gold, pred, K):
    f = []
    for c in range(K):
        tp = int(((gold == c) & (pred == c)).sum()); fp = int(((gold != c) & (pred == c)).sum()); fn = int(((gold == c) & (pred != c)).sum())
        p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
        f.append(2 * p * r / (p + r) if p + r else 0.0)
    return float(np.mean(f))

@torch.no_grad()
def evaluate(model, task, split):
    model.eval(); t = TASK_ID[task]; recs = DATA[task][split]; K = len(TARGETS[t])
    seqs = []
    for i, r in enumerate(recs):
        p = cached_prompt(r)
        for k, a in enumerate(TARGETS[t]):
            pp = p[: CFG["max_len"] - len(a)]
            seqs.append((i, k, dict(ids=pp + a, tgt=[0] * len(pp) + [1] * len(a))))
    order = sorted(range(len(seqs)), key=lambda j: len(seqs[j][2]["ids"]))
    scores = np.full((len(recs), K), -np.inf); bs = CFG["eval_batch_size"]
    for s in range(0, len(order), bs):
        chunk = [seqs[j] for j in order[s:s + bs]]
        ids, att, tgt = collate([c[2] for c in chunk])
        lp, rows = target_logprobs(model, ids, att, tgt)
        sums = torch.zeros(len(chunk), device=DEVICE).index_add_(0, rows, lp)
        cnt = torch.zeros(len(chunk), device=DEVICE).index_add_(0, rows, torch.ones_like(lp))
        for (i, k, _), v in zip(chunk, (sums / cnt).cpu().numpy()): scores[i, k] = v
    pred = scores.argmax(1); gold = np.array([r["label"] for r in recs])
    return dict(acc=float((pred == gold).mean()), macro_f1=macro_f1(gold, pred, K), preds=pred.tolist())

## 5. Training loop, class-stratified replay memory, online EWC on LoRA parameters

In [ ]:
def train_on(model, records, seed, ewc=None):
    set_seed(seed); model.train()
    P = list(trainable(model).values())
    opt = torch.optim.AdamW(P, lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    bs = CFG["batch_size"]; steps = CFG["epochs"] * math.ceil(len(records) / bs); warm = round(CFG["warmup_ratio"] * steps)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if warm and s < warm else max(0.0, (steps - s) / max(steps - warm, 1)))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    if DEVICE == "cuda": torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    rng = random.Random(seed); t0 = time.time(); losses, pens = [], []
    for _ in range(CFG["epochs"]):
        order = list(range(len(records))); rng.shuffle(order)
        for s in range(0, len(order), bs):
            loss = lm_loss(model, [records[j] for j in order[s:s + bs]]); total = loss
            if ewc is not None and ewc.F is not None:
                pen = ewc.penalty(model); total = loss + pen; pens.append(float(pen))
            opt.zero_grad(set_to_none=True)
            scaler.scale(total).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(P, CFG["grad_clip"])
            scaler.step(opt); scaler.update(); sch.step(); losses.append(float(loss))
    if DEVICE == "cuda": torch.cuda.synchronize()
    return dict(train_time_s=round(time.time() - t0, 2), optimizer_steps=steps, examples=len(records),
                peak_gpu_mem_MB=round(torch.cuda.max_memory_allocated() / 2**20, 1) if DEVICE == "cuda" else None,
                peak_gpu_reserved_MB=round(torch.cuda.max_memory_reserved() / 2**20, 1) if DEVICE == "cuda" else None,
                loss_first=round(float(np.mean(losses[:5])), 4), loss_last=round(float(np.mean(losses[-5:])), 4),
                ewc_penalty_mean=round(float(np.mean(pens)), 5) if pens else None, loss_curve=[round(x, 4) for x in losses])

class ReplayMemory:
    """Repository rule: target = round(fraction × n) per completed task, equal allocation per class, leftovers fill."""
    def __init__(self, fraction): self.fraction, self.by_task = fraction, {}
    def add_task(self, task, recs, seed):
        rng = random.Random(seed); groups = defaultdict(list)
        for r in recs: groups[r["label"]].append(r)
        labels = sorted(groups); target = max(1, round(len(recs) * self.fraction)); base, rem = divmod(target, len(labels))
        sel, left = [], []
        for pos, y in enumerate(labels):
            c = list(groups[y]); rng.shuffle(c); a = base + (1 if pos < rem else 0); sel += c[:a]; left += c[a:]
        if len(sel) < target: rng.shuffle(left); sel += left[: target - len(sel)]
        rng.shuffle(sel); self.by_task[task] = sel[:target]
    def all(self): return [r for v in self.by_task.values() for r in v]
    def storage_bytes(self):
        return len(json.dumps([dict(record_id=r["record_id"], label=r["label"], text=r["text"]) for r in self.all()],
                              ensure_ascii=False, sort_keys=True).encode("utf-8"))
    def class_counts(self): return {t: dict(Counter(r["label_name"] for r in v)) for t, v in self.by_task.items()}

class OnlineEWC:
    def __init__(self, lam): self.lam, self.F, self.star = lam, None, None
    def consolidate(self, model, recs, n, seed):
        t0 = time.time(); model.eval()
        sample = random.Random(seed).sample(list(recs), min(n, len(recs)))
        P = trainable(model); F = {k: torch.zeros_like(p, dtype=torch.float32) for k, p in P.items()}; S = 1024.0
        for r in sample:
            model.zero_grad(set_to_none=True); (lm_loss(model, [r]) * S).backward()
            for k, p in P.items():
                if p.grad is not None: F[k] += (p.grad.float() / S) ** 2
        model.zero_grad(set_to_none=True)
        F = {k: f / len(sample) for k, f in F.items()}
        mean = (sum(f.sum() for f in F.values()) / sum(f.numel() for f in F.values())).clamp_min(1e-30)
        F = {k: f / mean for k, f in F.items()}
        self.F = F if self.F is None else {k: self.F[k] + F[k] for k in F}
        self.star = {k: p.detach().clone().float() for k, p in P.items()}
        if DEVICE == "cuda": torch.cuda.synchronize()
        return dict(fisher_time_s=round(time.time() - t0, 2), fisher_samples=len(sample),
                    ewc_state_bytes=int(sum(v.numel() * v.element_size() for d in (self.F, self.star) for v in d.values())))
    def penalty(self, model):
        tot = 0.0
        for k, p in trainable(model).items(): tot = tot + (self.F[k] * (p.float() - self.star[k]) ** 2).sum()
        return 0.5 * self.lam * tot

## 6. Run schedule (19 runs) and runner — every run is saved immediately and skipped on restart

In [ ]:
T = CFG["tasks"]
def token(v): return str(v).replace(".", "p")
SCHEDULE = [dict(run_id=f"ewc_calibration_lambda_{token(l)}", method="ewc", seed=CFG["calibration_seed"], mode="calibration",
                 split="validation", lam=l) for l in CFG["ewc_lambda_grid"]]
for s in CFG["seeds"]:
    for m in ("sequential_lora", "experience_replay", "ewc", "joint_training"):
        SCHEDULE.append(dict(run_id=f"{m}_seed{s}", method=m, seed=s, mode="main", split="test"))
for f in CFG["sensitivity_fractions"]:
    SCHEDULE.append(dict(run_id=f"replay_sensitivity_{round(f * 100)}pct", method="experience_replay", seed=CFG["sensitivity_seed"],
                         mode="sensitivity", split="test", fraction=f))
print(len(SCHEDULE), "runs:", [r["run_id"] for r in SCHEDULE])

def path(run_id): return f"{RES}/runs/{run_id}.json"
ZS = {}
def zero_shot(split):
    if split in ZS: return ZS[split]
    p = f"{RES}/runs/zero_shot_{split}.json"
    if os.path.exists(p): ZS[split] = json.load(open(p)); return ZS[split]
    model = load_model(0); r = {t: evaluate(model, t, split) for t in T}
    json.dump(r, open(p, "w")); del model
    if DEVICE == "cuda": torch.cuda.empty_cache()
    ZS[split] = r; return r

def execute(spec, lam=None):
    if os.path.exists(path(spec["run_id"])):
        print("skip (done):", spec["run_id"]); return json.load(open(path(spec["run_id"])))
    print(f"\n=== {spec['run_id']} ===", flush=True); t_run = time.time()
    split, seed, method = spec["split"], spec["seed"], spec["method"]
    model = load_model(seed)
    n_tr, n_all = sum(p.numel() for p in trainable(model).values()), sum(p.numel() for p in model.parameters())
    out = dict(spec=spec, n_trainable_params=n_tr, n_total_params=n_all, env=ENV, lam=lam)
    if method == "joint_training":
        st = train_on(model, [r for t in T for r in DATA[t]["train"]], seed * 100 + 99)
        t0 = time.time(); ev = {t: evaluate(model, t, split) for t in T}; st["eval_time_s"] = round(time.time() - t0, 2)
        out.update(final_acc=[ev[t]["acc"] for t in T], final_f1=[ev[t]["macro_f1"] for t in T], stages=[st],
                   preds=[{t: ev[t]["preds"] for t in T}])
        print("  joint acc:", [round(a, 4) for a in out["final_acc"]], f"| train {st['train_time_s']}s | peak {st['peak_gpu_mem_MB']} MB")
    else:
        zs = zero_shot(split)
        acc = [[zs[t]["acc"] for t in T]]; f1 = [[zs[t]["macro_f1"] for t in T]]; preds = [{t: zs[t]["preds"] for t in T}]
        mem = ReplayMemory(spec.get("fraction", CFG["replay_fraction"])) if method == "experience_replay" else None
        ewc = OnlineEWC(lam) if method == "ewc" else None; stages = []
        for i, task in enumerate(T):
            replay = mem.all() if mem else []
            st = train_on(model, DATA[task]["train"] + replay, seed * 100 + i, ewc=ewc); st.update(task=task, replay_examples_used=len(replay))
            if mem:
                mem.add_task(task, DATA[task]["train"], seed * 100 + i)
                st.update(replay_memory_examples=len(mem.all()), replay_storage_bytes=mem.storage_bytes(), replay_class_counts=mem.class_counts())
            if ewc: st.update(ewc.consolidate(model, DATA[task]["train"], CFG["ewc_fisher_samples"], seed * 100 + i))
            t0 = time.time(); ev = {t: evaluate(model, t, split) for t in T}; st["eval_time_s"] = round(time.time() - t0, 2)
            acc.append([ev[t]["acc"] for t in T]); f1.append([ev[t]["macro_f1"] for t in T]); preds.append({t: ev[t]["preds"] for t in T})
            stages.append(st)
            print(f"  after {task}: acc={[round(a, 4) for a in acc[-1]]} | train {st['train_time_s']}s | peak {st['peak_gpu_mem_MB']} MB", flush=True)
        out.update(acc_matrix=acc, f1_matrix=f1, stages=stages, preds=preds)
    out["run_wall_time_s"] = round(time.time() - t_run, 1); out["finished_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    json.dump(out, open(path(spec["run_id"]) + ".tmp", "w")); os.replace(path(spec["run_id"]) + ".tmp", path(spec["run_id"]))
    del model
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return out

def cl_metrics(R):
    R = np.array(R)[1:]; n = R.shape[0]
    return dict(ACC=float(R[-1].mean()), LA=float(np.mean([R[i, i] for i in range(n)])),
                BWT=float(np.mean([R[-1, j] - R[j, j] for j in range(n - 1)])),
                FGT=float(np.mean([R[j:n - 1, j].max() - R[-1, j] for j in range(n - 1)])))

## 7. EWC λ calibration on validation data (seed 1000) → selected λ

In [ ]:
calib = []
for spec in [s for s in SCHEDULE if s["mode"] == "calibration"]:
    r = execute(spec, lam=spec["lam"]); m = cl_metrics(r["acc_matrix"]); calib.append(dict(lam=spec["lam"], **m))
    print(f"λ={spec['lam']}: val ACC={m['ACC']:.4f} FGT={m['FGT']:.4f} LA={m['LA']:.4f}")
best = max(calib, key=lambda d: (round(d["ACC"], 6), -d["FGT"], -d["lam"]))
LAMBDA = best["lam"]
json.dump(dict(calibration=calib, selected_lambda=LAMBDA, rule="highest validation ACC, then lower FGT, then lower λ"),
          open(f"{RES}/tables/ewc_calibration.json", "w"), indent=2)
print("selected λ =", LAMBDA)

## 8. Main runs (4 conditions × seeds 0, 1, 2) and replay sensitivity (5 %, 20 %, seed 0)

In [ ]:
RUNS = {}
for spec in [s for s in SCHEDULE if s["mode"] in ("main", "sensitivity")]:
    RUNS[spec["run_id"]] = execute(spec, lam=LAMBDA if spec["method"] == "ewc" else None)
json.dump(dict(CFG, selected_lambda=LAMBDA), open(f"{RES}/config.json", "w"), indent=2)
print("all runs finished")

## 9. Tables: main comparison, per-task final accuracy, per-run metrics, resources

In [ ]:
import pandas as pd
MN = {"sequential_lora": "Seq-LoRA", "experience_replay": "ER (10%)", "ewc": "Online EWC", "joint_training": "Joint"}
METHODS = ["sequential_lora", "experience_replay", "ewc", "joint_training"]
rows = []
for s in CFG["seeds"]:
    for m in METHODS:
        r = RUNS[f"{m}_seed{s}"]; st = r["stages"]
        row = dict(run_id=f"{m}_seed{s}", method=m, seed=s)
        if m == "joint_training":
            row.update(ACC=float(np.mean(r["final_acc"])), macro_F1=float(np.mean(r["final_f1"])),
                       **{f"acc_{t}": r["final_acc"][j] for j, t in enumerate(T)})
        else:
            row.update(cl_metrics(r["acc_matrix"]), macro_F1=float(np.mean(r["f1_matrix"][-1])),
                       **{f"acc_{t}": r["acc_matrix"][-1][j] for j, t in enumerate(T)})
        row.update(train_time_min=sum(x["train_time_s"] for x in st) / 60, fisher_time_min=sum(x.get("fisher_time_s", 0) for x in st) / 60,
                   eval_time_min=sum(x["eval_time_s"] for x in st) / 60, run_wall_min=r["run_wall_time_s"] / 60,
                   optimizer_steps=sum(x["optimizer_steps"] for x in st), peak_gpu_mem_MB=max((x["peak_gpu_mem_MB"] or 0) for x in st),
                   extra_state_bytes=max([x.get("replay_storage_bytes", 0) for x in st] + [x.get("ewc_state_bytes", 0) for x in st]),
                   trainable_params=r["n_trainable_params"], total_params=r["n_total_params"])
        rows.append(row)
df = pd.DataFrame(rows); df.to_csv(f"{RES}/tables/per_run_metrics.csv", index=False)
num = [c for c in df.columns if c not in ("run_id", "method", "seed")]
agg = df.groupby("method")[num].agg(["mean", lambda x: x.std(ddof=1)]).reindex(METHODS)
agg.columns = [f"{a}_{'mean' if b == 'mean' else 'sd'}" for a, b in agg.columns]; agg.to_csv(f"{RES}/tables/summary_mean_sd.csv")
show = pd.DataFrame({k: [f"{100*agg.loc[m, k+'_mean']:.2f} ± {100*agg.loc[m, k+'_sd']:.2f}" if pd.notna(agg.loc[m, k + "_mean"]) else "—" for m in METHODS]
                     for k in ["ACC", "macro_F1", "FGT", "LA", "BWT"] + [f"acc_{t}" for t in T]}, index=[MN[m] for m in METHODS])
show.to_csv(f"{RES}/tables/main_comparison_pretty.csv")
print("selected λ:", LAMBDA, "| trainable params:", df.trainable_params.iloc[0], "of", df.total_params.iloc[0])
print("zero-shot test acc:", {t: round(zero_shot("test")[t]["acc"], 4) for t in T})
show

In [ ]:
sens = [dict(fraction=CFG["replay_fraction"], **{k: v for k, v in rows[[x["run_id"] for x in rows].index(f"experience_replay_seed{CFG['sensitivity_seed']}")].items()
                                                  if k in ("ACC", "FGT", "LA", "BWT", "train_time_min", "optimizer_steps", "extra_state_bytes", "macro_F1")})]
for f in CFG["sensitivity_fractions"]:
    r = RUNS[f"replay_sensitivity_{round(f * 100)}pct"]; st = r["stages"]
    sens.append(dict(fraction=f, **cl_metrics(r["acc_matrix"]), macro_F1=float(np.mean(r["f1_matrix"][-1])),
                     train_time_min=sum(x["train_time_s"] for x in st) / 60, optimizer_steps=sum(x["optimizer_steps"] for x in st),
                     extra_state_bytes=max(x.get("replay_storage_bytes", 0) for x in st)))
sens = pd.DataFrame(sens).sort_values("fraction"); sens.to_csv(f"{RES}/tables/replay_sensitivity.csv", index=False); sens

## 10. Statistics vs Seq-LoRA: paired bootstrap (10,000 item resamples) and exact McNemar with Holm correction

In [ ]:
from scipy.stats import binomtest
gold = {t: np.array([r["label"] for r in DATA[t]["test"]]) for t in T}
def correct(run, stage=-1): return {t: (np.array(run["preds"][stage][t]) == gold[t]).astype(float) for t in T}
C = {(m, s): correct(RUNS[f"{m}_seed{s}"]) for m in METHODS for s in CFG["seeds"]}
rng_b = np.random.default_rng(CFG["stats_seed"]); B = CFG["bootstrap_resamples"]
IDX = {t: rng_b.integers(0, len(gold[t]), size=(B, len(gold[t]))) for t in T}
boot = []
for m in ["experience_replay", "ewc", "joint_training"]:
    for scope, tasks in [("AG News", [T[0]]), ("SST-2", [T[1]]), ("Emotion", [T[2]]), ("Earlier tasks (mean)", T[:2]), ("All tasks (ACC)", T)]:
        dist = np.zeros(B); point = []
        for s in CFG["seeds"]:
            a, b = C[(m, s)], C[("sequential_lora", s)]
            dist += np.mean([(a[t][IDX[t]] - b[t][IDX[t]]).mean(1) for t in tasks], axis=0)
            point.append(np.mean([a[t].mean() - b[t].mean() for t in tasks]))
        dist /= len(CFG["seeds"]); lo, hi = np.quantile(dist, [0.025, 0.975])
        boot.append(dict(method=MN[m], scope=scope, delta_pp=100 * float(np.mean(point)), ci_low_pp=100 * float(lo), ci_high_pp=100 * float(hi),
                         seeds_positive=int(sum(p > 0 for p in point))))
boot = pd.DataFrame(boot); boot.to_csv(f"{RES}/tables/bootstrap_vs_seq.csv", index=False)

def holm(ps):
    o = np.argsort(ps); adj = np.empty(len(ps)); run = 0
    for i, j in enumerate(o): run = max(run, min(1, (len(ps) - i) * ps[j])); adj[j] = run
    return adj
mc = []
for m in ["experience_replay", "ewc"]:
    fam = []
    for s in CFG["seeds"]:
        for t in T[:2]:
            a, b = C[(m, s)][t], C[("sequential_lora", s)][t]
            n01, n10 = int(((a == 1) & (b == 0)).sum()), int(((a == 0) & (b == 1)).sum())
            fam.append(dict(method=MN[m], seed=s, task=t, method_only_correct=n01, seq_only_correct=n10,
                            p=binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0))
    for d, pa in zip(fam, holm([d["p"] for d in fam])): d["p_holm_within_method"] = float(pa)
    mc += fam
mc = pd.DataFrame(mc); mc.to_csv(f"{RES}/tables/mcnemar_per_seed.csv", index=False)
pooled = []
for m in ["experience_replay", "ewc"]:
    fam = []
    for t in T[:2]:
        n01 = int(sum(((C[(m, s)][t] == 1) & (C[("sequential_lora", s)][t] == 0)).sum() for s in CFG["seeds"]))
        n10 = int(sum(((C[(m, s)][t] == 0) & (C[("sequential_lora", s)][t] == 1)).sum() for s in CFG["seeds"]))
        fam.append(dict(method=MN[m], task=t, method_only_correct=n01, seq_only_correct=n10, p=binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0))
    for d, pa in zip(fam, holm([d["p"] for d in fam])): d["p_holm_within_method"] = float(pa)
    pooled += fam
pooled = pd.DataFrame(pooled); pooled.to_csv(f"{RES}/tables/mcnemar_pooled_over_seeds.csv", index=False)
print(boot.round(3).to_string()); print(); print(pooled.to_string()); print(); print(mc.to_string())

## 11. Error analysis data: confusion matrices, forgotten items, 30-error audit sample (10 per task)

In [ ]:
err = dict(confusion={}, forgotten={}, audit_sample=[])
for t in T:
    K = len(LABELS[TASK_ID[t]]); err["confusion"][t] = {}
    for m in METHODS:
        cm = np.zeros((K, K), dtype=int)
        for s in CFG["seeds"]:
            for g, p in zip(gold[t], RUNS[f"{m}_seed{s}"]["preds"][-1][t]): cm[g, p] += 1
        err["confusion"][t][m] = cm.tolist()
for m in ["sequential_lora", "experience_replay", "ewc"]:   # items correct right after learning the task, wrong at the end
    err["forgotten"][m] = {}
    for j, t in enumerate(T[:2]):
        n_forgot = n_learned = 0
        for s in CFG["seeds"]:
            r = RUNS[f"{m}_seed{s}"]; after = np.array(r["preds"][j + 1][t]) == gold[t]; final = np.array(r["preds"][-1][t]) == gold[t]
            n_learned += int(after.sum()); n_forgot += int((after & ~final).sum())
        err["forgotten"][m][t] = dict(correct_after_learning=n_learned, forgotten_at_end=n_forgot)
rng_e = random.Random(CFG["stats_seed"])
for t in T:   # final-stage errors of Seq-LoRA seed 0, fixed random sample of 10 per task for manual classification
    r = RUNS["sequential_lora_seed0"]; recs = DATA[t]["test"]; preds = r["preds"][-1][t]
    wrong = [i for i, (g, p) in enumerate(zip(gold[t], preds)) if g != p]
    for i in sorted(rng_e.sample(wrong, min(10, len(wrong)))):
        others = {MN[m]: LABELS[TASK_ID[t]][RUNS[f"{m}_seed0"]["preds"][-1][t][i]] for m in ["experience_replay", "ewc"]}
        err["audit_sample"].append(dict(task=t, record_id=recs[i]["record_id"], gold=recs[i]["label_name"],
                                        seq_pred=LABELS[TASK_ID[t]][preds[i]], other_methods_pred=others, text=recs[i]["text"]))
err["zero_shot_pred_distribution"] = {t: dict(Counter(LABELS[TASK_ID[t]][p] for p in zero_shot("test")[t]["preds"])) for t in T}
err["final_pred_distribution"] = {m: {t: dict(Counter(LABELS[TASK_ID[t]][p] for s in CFG["seeds"] for p in RUNS[f"{m}_seed{s}"]["preds"][-1][t]))
                                      for t in T} for m in METHODS}
json.dump(err, open(f"{RES}/tables/error_analysis.json", "w"), indent=2, ensure_ascii=False)
print(json.dumps(err["forgotten"], indent=1))

## 12. Quick-look figures

In [ ]:
import matplotlib; import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e4e3df"})
COL = {"sequential_lora": "#2a78d6", "experience_replay": "#eb6834", "ewc": "#1baf7a", "joint_training": "#8a8984"}
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for j, t in enumerate(T):
    for m in METHODS[:3]:
        M = np.array([RUNS[f"{m}_seed{s}"]["acc_matrix"] for s in CFG["seeds"]])[:, :, j] * 100
        axes[j].plot(range(4), M.mean(0), marker="o", lw=2, color=COL[m], label=MN[m])
        axes[j].fill_between(range(4), M.mean(0) - M.std(0, ddof=1), M.mean(0) + M.std(0, ddof=1), color=COL[m], alpha=.15, lw=0)
    axes[j].axhline(100 * df[df.method == "joint_training"][f"acc_{t}"].mean(), ls="--", color=COL["joint_training"], label="Joint")
    axes[j].set_title(t); axes[j].set_xticks(range(4)); axes[j].set_xticklabels(["base", "after T1", "after T2", "after T3"])
axes[0].set_ylabel("test accuracy (%)"); axes[-1].legend(frameon=False); fig.tight_layout()
fig.savefig(f"{RES}/figures/accuracy_trajectories.png", bbox_inches="tight"); plt.show()

## 13. Package results → `rmt_a4_real_results.zip` (send this file back)

In [ ]:
json.dump(dict(finished_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), measured_model_runs=not SMOKE_TEST,
               n_runs=len([f for f in os.listdir(f"{RES}/runs") if not f.startswith("zero_shot")]), env=ENV, selected_lambda=LAMBDA),
          open(f"{RES}/manifest.json", "w"), indent=2)
zpath = os.path.join(os.path.dirname(RES.rstrip("/")), "rmt_a4_real_results.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(RES):
        for f in files: z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), os.path.dirname(RES.rstrip("/"))))
print("saved", zpath, round(os.path.getsize(zpath) / 1e6, 2), "MB")
try:
    from google.colab import files; files.download(zpath)
except Exception: pass